# 🏓 Notebook 04: Player Detection Training**Dataset:** Pickleball with Players (4,119 images) — Roboflow**Model:** YOLO11m (medium)**Classes:** Person, Pickleball

## 1. Setup

In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics roboflow -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/pickleball_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print('✅ Drive mounted')


## 2. Download Dataset từ Roboflow

In [ ]:
from roboflow import Roboflow

# ===== CẤU HÌNH =====
ROBOFLOW_API_KEY = "VAwW4zaxVs978t7iLszZ"  # ← Thay API key của bạn
# =====================

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset 1: Pickleball with Players (4,119 images) - KHUYÊN DÙNG
project = rf.workspace("hughs-workspace-plw3g").project("pickleball-with-players-topw1")
dataset = project.version(1).download("yolov8", location="/content/dataset_players")

print(f'✅ Dataset downloaded: {dataset.location}')
print(f'📁 Train: {os.path.join(dataset.location, "train")}')


In [ ]:
# Kiểm tra dataset
import glob
train_imgs = glob.glob(os.path.join(dataset.location, 'train', 'images', '*'))
val_imgs = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*'))
test_imgs = glob.glob(os.path.join(dataset.location, 'test', 'images', '*'))
print(f'📊 Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}')

# Xem data.yaml
yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path) as f:
    print(f.read())


In [ ]:
# Xem sample images
import cv2
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, img_path in enumerate(train_imgs[:6]):
    ax = axes[idx//3][idx%3]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Draw labels if exist
    label_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(label_path):
        h, w = img.shape[:2]
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = (255,0,0) if cls == 0 else (0,255,0)
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                cv2.putText(img, f'cls{cls}', (x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()


## 3. Train Model

In [ ]:
from ultralytics import YOLO

# ===== CẤU HÌNH TRAINING =====
MODEL_SIZE = 'yolo11m.pt'  # Medium — cân bằng tốc độ/chính xác
EPOCHS = 80
BATCH = 32         # A100: tăng lên 32; T4: giữ 16
IMGSZ = 640
PATIENCE = 15      # Early stopping
# ==============================

model = YOLO(MODEL_SIZE)
print(f'🔧 Model: {MODEL_SIZE}')
print(f'📐 imgsz={IMGSZ}, batch={BATCH}, epochs={EPOCHS}')


In [ ]:
# TRAIN
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    workers=8,
    amp=True,           # Mixed precision (FP16)
    cache='ram',         # Cache images in RAM → faster
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5,
    translate=0.1,
    scale=0.3,
    flipud=0.0,
    fliplr=0.5,
    name='player_detection',
    project='/content/runs',
    exist_ok=True,
    verbose=True,
)
print('✅ Training complete!')


## 4. Evaluate

In [ ]:
# Validation metrics
metrics = model.val()
print(f'\n📊 Validation Results:')
print(f'  mAP@50:    {metrics.box.map50:.3f}')
print(f'  mAP@50-95: {metrics.box.map:.3f}')
print(f'  Precision:  {metrics.box.mp:.3f}')
print(f'  Recall:     {metrics.box.mr:.3f}')


In [ ]:
# Training curves
from IPython.display import Image, display
import os

run_dir = '/content/runs/player_detection'
curves = ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png',
          'PR_curve.png', 'F1_curve.png']
for c in curves:
    p = os.path.join(run_dir, c)
    if os.path.exists(p):
        print(f'\n📈 {c}:')
        display(Image(filename=p, width=800))


In [ ]:
# Test on sample images
test_results = model.predict(
    source=os.path.join(dataset.location, 'test', 'images'),
    conf=0.5,
    save=True,
    project='/content/runs',
    name='player_test_predict',
    exist_ok=True,
)

# Show predictions
pred_dir = '/content/runs/player_test_predict'
pred_imgs = sorted(glob.glob(os.path.join(pred_dir, '*.jpg')))[:6]
if pred_imgs:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for idx, p in enumerate(pred_imgs):
        ax = axes[idx//3][idx%3]
        img = cv2.imread(p)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(os.path.basename(p), fontsize=9)
        ax.axis('off')
    plt.suptitle('Test Predictions', fontsize=14)
    plt.tight_layout()
    plt.show()


## 5. Save Model

In [ ]:
import shutil

# Best weights
best_pt = '/content/runs/player_detection/weights/best.pt'
dest = os.path.join(SAVE_DIR, 'player_detection_best.pt')
shutil.copy2(best_pt, dest)
print(f'✅ Model saved: {dest}')

# Also save last
last_pt = '/content/runs/player_detection/weights/last.pt'
if os.path.exists(last_pt):
    shutil.copy2(last_pt, os.path.join(SAVE_DIR, 'player_detection_last.pt'))
    print(f'✅ Last saved: {os.path.join(SAVE_DIR, "player_detection_last.pt")}')


In [ ]:
# Quick test on video (optional)
# model_test = YOLO(dest)
# results = model_test.predict('/content/drive/MyDrive/pickleball_match.mp4',
#     conf=0.5, save=True, project='/content/runs', name='player_video_test')


## 6. Tích hợp vào PipelineSau khi train xong, sửa notebook `03_pipeline_inference.ipynb`:```python# Thay dòng:self.model = YOLO('yolo11n.pt')  # pretrained# Thành:PLAYER_MODEL = '/content/drive/MyDrive/pickleball_models/player_detection_best.pt'self.model = YOLO(PLAYER_MODEL)  # custom trained# Và bỏ classes=[0] vì model custom đã chỉ detect player```